# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import os
import requests
import queue
import threading
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from fake_useragent import UserAgent

# --- 사용자 설정 ---
INITIAL_START_ID = 6654626
END_ID = 1
NUM_THREADS = 10
CHUNK_SIZE = 1000   # 파일당 저장 건수 (1000.parquet, 2000.parquet, ...)
SAVE_DIR = r"../data"

# --- 전역 변수 및 락 ---
buffer = []
next_chunk_num = 1   # 다음에 저장할 청크 번호 (1 → 1000.parquet, 2 → 2000.parquet, ...)
save_lock = threading.Lock()
task_queue = queue.Queue(maxsize=2000)
ua = UserAgent()

# ───────────────────────────────────────────────
# 재시작 정보 계산
# ───────────────────────────────────────────────
def get_parquet_files():
    """SAVE_DIR 안의 숫자 이름 parquet 파일 목록을 숫자 순으로 반환"""
    files = [f for f in os.listdir(SAVE_DIR)
             if f.endswith('.parquet') and f.replace('.parquet', '').isdigit()]
    return sorted(files, key=lambda x: int(x.replace('.parquet', '')))

def get_resume_info():
    """기존 파일에서 (시작 ID, 다음 청크 번호, 수집된 ID 집합)을 반환"""
    files = get_parquet_files()
    if not files:
        return INITIAL_START_ID, 1, set()

    existing_ids = set()
    for f in files:
        df_tmp = pd.read_parquet(os.path.join(SAVE_DIR, f), columns=['id'])
        existing_ids.update(df_tmp['id'].values)

    last_num = int(files[-1].replace('.parquet', ''))
    next_num = last_num // CHUNK_SIZE + 1
    resume_id = int(min(existing_ids)) - 1
    print(f"기존 파일 {len(files)}개, {len(existing_ids):,}건 확인.")
    print(f"ID {resume_id}부터 이어서 시작합니다. (다음 파일: {next_num * CHUNK_SIZE}.parquet)")
    return resume_id, next_num, existing_ids

# ───────────────────────────────────────────────
# 청크 저장
# ───────────────────────────────────────────────
def try_save_chunk(pbar=None):
    """버퍼가 CHUNK_SIZE 이상이면 잘라서 파일로 저장 (lock 내부에서 호출)"""
    global buffer, next_chunk_num
    while len(buffer) >= CHUNK_SIZE:
        chunk = buffer[:CHUNK_SIZE]
        buffer = buffer[CHUNK_SIZE:]
        filename = f'{next_chunk_num * CHUNK_SIZE}.parquet'
        pd.DataFrame(chunk).to_parquet(os.path.join(SAVE_DIR, filename), index=False)
        tqdm.write(f'저장: {filename}')
        next_chunk_num += 1

def save_remaining(pbar=None):
    """종료 시 버퍼에 남은 데이터를 실제 건수 이름으로 저장"""
    global buffer, next_chunk_num
    with save_lock:
        if not buffer:
            return
        saved_so_far = (next_chunk_num - 1) * CHUNK_SIZE
        total = saved_so_far + len(buffer)
        filename = f'{total}.parquet'
        pd.DataFrame(buffer).to_parquet(os.path.join(SAVE_DIR, filename), index=False)
        tqdm.write(f'저장: {filename} (잔여 {len(buffer)}건)')
        buffer = []

# ───────────────────────────────────────────────
# API 요청
# ───────────────────────────────────────────────
def fetch_api_data(post_id):
    url = f"https://safebooru.org/index.php?page=dapi&s=post&q=index&id={post_id}"
    headers = {'User-Agent': ua.random}
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            post = root.find('post')
            if post is not None:
                file_url = post.get('file_url', '')
                if file_url and not file_url.startswith('http'):
                    file_url = 'https:' + file_url
                return {
                    'id':         int(post.get('id')),
                    'tags':       post.get('tags'),
                    'file_url':   file_url,
                    'sample_url': post.get('sample_url', ''),
                    'width':      int(post.get('width', 0)),
                    'height':     int(post.get('height', 0)),
                }
    except Exception:
        pass
    return None

# ───────────────────────────────────────────────
# 워커 스레드
# ───────────────────────────────────────────────
def worker(pbar, existing_ids):
    while True:
        post_id = task_queue.get()
        if post_id is None:
            task_queue.task_done()
            break

        if post_id not in existing_ids:
            result = fetch_api_data(post_id)
            if result:
                with save_lock:
                    buffer.append(result)
                    try_save_chunk(pbar)

        pbar.update(1)
        task_queue.task_done()

# ───────────────────────────────────────────────
# 메인 실행
# ───────────────────────────────────────────────
if __name__ == "__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)

    current_start_id, next_chunk_num, existing_ids = get_resume_info()
    total_tasks = current_start_id - END_ID + 1

    print(f"=== 크롤링 시작 (ID: {current_start_id} → {END_ID}, 스레드: {NUM_THREADS}) ===")

    pbar = tqdm(total=total_tasks, desc="수집 진행률", ncols=100)

    threads = []
    for _ in range(NUM_THREADS):
        t = threading.Thread(target=worker, args=(pbar, existing_ids))
        t.daemon = True
        t.start()
        threads.append(t)

    try:
        for pid in range(current_start_id, END_ID - 1, -1):
            task_queue.put(pid)
        task_queue.join()
    except KeyboardInterrupt:
        print("\n[!] 중단 요청. 남은 버퍼를 저장하고 종료합니다.")

    for _ in range(NUM_THREADS):
        task_queue.put(None)
    for t in threads:
        t.join()

    save_remaining(pbar)
    pbar.close()

    files = get_parquet_files()
    print(f"\n=== 완료: 파일 {len(files)}개 저장됨 ===")